# 🌸 Pipeline de Classificação Botânica (Oxford Flowers-102)
## Execução Auditada e Reproduzível no Google Colab Pro (GPU Tesla T4)

Este notebook executa o pipeline científico revisado de classificação botânica com a rede **MobileNetV2** no benchmark oficial **Oxford Flowers-102** ($8.189$ imagens em $102$ classes).

### 🛡️ Princípios Metodológicos Rigorosos Desta Versão:
1. **Zero fabricação de dados:** Nenhuma métrica forçada via `max()`, deltas parciais extrapolados ou $p$-valores fixados manualmente.
2. **Validação Estatística Pareada:** Teste de McNemar com correção de continuidade de Edwards e Bootstrap pareado ($B=1000$) para o ganho do fine-tuning.
3. **Preservação de Aspect-Ratio:** Redimensionamento isotrópico (Resize 256 + CenterCrop 224) eliminando distorções ópticas nas corolas florais.
4. **Congelamento Estrito de BatchNorm:** Camadas de normalização do backbone mantidas em `eval()` durante a Fase 1.
5. **Calibração Probabilística Auditável:** Cálculo de ECE com tratamento seguro de limites ($p \le 1.0$) e bins vazios, complementado por Brier Score e Negative Log-Likelihood (NLL).
6. **Persistência Completa de Artefatos:** Salva predições brutas (`predictions.npz`), histórico (`history.json`), métricas (`metrics.json`), checkpoint e relatório formal DOCX.

### 1. Diagnóstico de Hardware e Ambiente
Verifica a GPU alocada pelo Google Colab Pro (ex: Tesla T4 de 16 GB), versão do CUDA e VRAM disponível.

In [ ]:
!nvidia-smi
import torch, platform
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 2. Clonagem do Repositório e Instalação de Dependências

In [ ]:
# Caso execute a partir do GitHub:
# !git clone https://github.com/kelryana/classificacao-botanica.git
# %cd classificacao-botanica
# !git checkout revisao-metodologica

!pip install -q -r requirements.txt

### 3. Execução da Suíte de Testes de Software
Executa a bateria de testes unitários que valida a matemática de métricas, calibração ECE, McNemar e invariância de BatchNorm.

In [ ]:
!pytest test_pipeline.py -v

### 4. Execução do Pipeline Científico de Ponta a Ponta
O script executa:
- Download dos 3 splits oficiais do Oxford Flowers-102 (1.020 Treino / 1.020 Validação / 6.149 Teste);
- Extração determinística de características e benchmarking clássico (KNN, Random Forest, Regressão Logística);
- Fase 1 (Cabeça MLP com backbone e BatchNorm congelados);
- Fase 2 (Fine-tuning progressivo dos blocos features.16-18);
- Avaliação de TTA (Test-Time Augmentation) no teste completo;
- Cálculo de McNemar, Bootstrap pareado, calibração e Grad-CAM;
- Geração das 11 figuras técnicas em 300 DPI e do relatório formal DOCX.

In [ ]:
!python pipeline_botanico.py --base-dir ./resultados_botanico --data-dir ./data --epochs-p1 8 --epochs-p2 4 --batch-size 32

### 5. Verificação e Rastreabilidade das Métricas Gravadas

In [ ]:
import json
with open("./resultados_botanico/metrics.json", "r", encoding="utf-8") as f:
    res = json.load(f)

print("=" * 60)
print("MÉTRICAS VERIFICADAS DO MODELO FINAL (TESTE - 6.149 AMOSTRAS)")
print("=" * 60)
print(f"Top-1 Accuracy: {res['metrics_ft']['top1']:.2f}%")
print(f"Top-5 Accuracy: {res['metrics_ft']['top5']:.2f}%")
print(f"Macro F1-Score: {res['metrics_ft']['f1_macro']:.2f}%")
print(f"Expected Calibration Error (ECE): {res['calibration']['ece_percentage']:.2f}%")
print(f"Brier Score: {res['calibration']['brier_score']:.4f}")
print(f"Negative Log-Likelihood (NLL): {res['calibration']['nll']:.4f}")
print(f"McNemar chi2: {res['mcnemar_ft']['chi2']:.2f} (p = {res['mcnemar_ft']['p_value']:.4e})")
print(f"Bootstrap IC 95%: [{res['boot_ci_ft']['ci_low']:.2f}%, {res['boot_ci_ft']['ci_high']:.2f}%]")
if res.get('metrics_tta'):
    print(f"Com TTA (Full Test): Top-1 = {res['metrics_tta']['top1']:.2f}%")

### 6. Inspeção Visual das 11 Figuras em Alta Resolução (300 DPI)

In [ ]:
from pathlib import Path
plots = sorted(list(Path("./resultados_botanico/plots").glob("*.png")))
print(f"Total de figuras geradas: {len(plots)}")
for p in plots:
    print(f"  -> {p.name}")